# Tablero climatico con Kafka y ZooKeeper

Notebook autocontenido para Colab que fusiona `tablero_clima_con_mapa_v2` con el pipeline Kafka de `BA_Weather_Kafka_Colab_Compacto`. El HTML final del tablero se conserva sin cambios: Kafka se agrega como capa de ingesta/enriquecimiento antes del render.


In [ ]:
# @title 1. Instalacion completa: Java, Kafka, ZooKeeper y librerias Python
import os
import shutil
import subprocess
import sys
from pathlib import Path

KAFKA_VERSION = "3.9.2"          # Kafka 3.x conserva el modo ZooKeeper; Kafka 4.x ya es KRaft.
SCALA_VERSION = "2.13"
KAFKA_DIR = Path("/content/kafka")
KAFKA_TGZ = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_URLS = [
    f"https://downloads.apache.org/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}",
    f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}",
]
TOPIC_NAME = "tablero-climatico-ciudades"
BOOTSTRAP_SERVERS = "localhost:9092"

print("Instalando dependencias del sistema...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless", "wget"], check=True)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("Instalando librerias Python...")
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "kafka-python",
    "requests",
    "pandas",
    "ipywidgets",
], check=True)

if not (KAFKA_DIR / "bin" / "kafka-server-start.sh").exists():
    print(f"Descargando Apache Kafka {KAFKA_VERSION}...")
    shutil.rmtree(KAFKA_DIR, ignore_errors=True)
    tgz_path = Path("/content") / KAFKA_TGZ
    if tgz_path.exists():
        tgz_path.unlink()

    downloaded = False
    for url in KAFKA_URLS:
        print(f"Probando: {url}")
        result = subprocess.run(["wget", "-q", url, "-O", str(tgz_path)], check=False)
        if result.returncode == 0 and tgz_path.exists() and tgz_path.stat().st_size > 0:
            downloaded = True
            break
    if not downloaded:
        raise RuntimeError("No se pudo descargar Kafka desde downloads.apache.org ni archive.apache.org")

    subprocess.run(["tar", "-xzf", str(tgz_path), "-C", "/content"], check=True)
    extracted_dir = Path("/content") / f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
    if KAFKA_DIR.exists():
        shutil.rmtree(KAFKA_DIR)
    shutil.move(str(extracted_dir), str(KAFKA_DIR))
else:
    print(f"Kafka ya esta instalado en {KAFKA_DIR}")

import json
import requests
import pandas as pd
from IPython.display import HTML, display

print("Environment listo")
print(f"Kafka: {KAFKA_DIR}")
print(f"Topic configurado: {TOPIC_NAME}")


In [ ]:
# @title 2. Levantar ZooKeeper, Kafka y crear el topic
import os
import shutil
import socket
import subprocess
import time
from pathlib import Path

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]


def wait_for_port(host, port, timeout=60):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=2):
                return True
        except OSError as exc:
            last_error = exc
            time.sleep(1)
    raise TimeoutError(f"Puerto {host}:{port} no disponible tras {timeout}s: {last_error}")


def stop_kafka_stack():
    if (KAFKA_DIR / "bin" / "kafka-server-stop.sh").exists():
        subprocess.run([str(KAFKA_DIR / "bin" / "kafka-server-stop.sh")], check=False)
    if (KAFKA_DIR / "bin" / "zookeeper-server-stop.sh").exists():
        subprocess.run([str(KAFKA_DIR / "bin" / "zookeeper-server-stop.sh")], check=False)
    # En Colab a veces quedan procesos Java huerfanos al re-ejecutar celdas.
    subprocess.run("jps | awk '/Kafka|QuorumPeerMain/ {print $1}' | xargs -r kill -TERM", shell=True, check=False)
    time.sleep(4)


stop_kafka_stack()
shutil.rmtree("/tmp/kafka-logs", ignore_errors=True)
shutil.rmtree("/tmp/zookeeper", ignore_errors=True)

zk_log = open("/content/zookeeper.log", "w")
kafka_log = open("/content/kafka.log", "w")

zookeeper_proc = subprocess.Popen(
    [str(KAFKA_DIR / "bin" / "zookeeper-server-start.sh"), str(KAFKA_DIR / "config" / "zookeeper.properties")],
    stdout=zk_log,
    stderr=subprocess.STDOUT,
)
wait_for_port("localhost", 2181, timeout=45)
print("ZooKeeper listo en localhost:2181")

kafka_proc = subprocess.Popen(
    [str(KAFKA_DIR / "bin" / "kafka-server-start.sh"), str(KAFKA_DIR / "config" / "server.properties")],
    stdout=kafka_log,
    stderr=subprocess.STDOUT,
)
wait_for_port("localhost", 9092, timeout=75)
print("Kafka broker listo en localhost:9092")

subprocess.run([
    str(KAFKA_DIR / "bin" / "kafka-topics.sh"),
    "--create",
    "--if-not-exists",
    "--topic",
    TOPIC_NAME,
    "--bootstrap-server",
    BOOTSTRAP_SERVERS,
    "--partitions",
    "1",
    "--replication-factor",
    "1",
], check=True)

subprocess.run([
    str(KAFKA_DIR / "bin" / "kafka-topics.sh"),
    "--describe",
    "--topic",
    TOPIC_NAME,
    "--bootstrap-server",
    BOOTSTRAP_SERVERS,
], check=True)

print(f"Kafka + ZooKeeper funcionando. Topic: {TOPIC_NAME}")
print("Logs: /content/zookeeper.log y /content/kafka.log")


In [ ]:
# @title
"""Celda de Introducción del Panel Meteorológico.
Genera un encabezado estilizado, responsivo y con soporte dark-mode.
"""

from IPython.display import HTML, display

intro_html = """
<style>
  :root {
    --intro-bg: linear-gradient(135deg, #f8fafc 0%, #e2e8f0 100%);
    --intro-card: rgba(255, 255, 255, 0.9);
    --intro-text: #0f172a;
    --intro-muted: #475569;
    --intro-border: #cbd5e1;
    --badge-bg: #f1f5f9;
    --badge-border: #e2e8f0;
    --badge-text: #334155;
    --accent: #3b82f6;
  }

  @media (prefers-color-scheme: dark) {
    :root {
      --intro-bg: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
      --intro-card: rgba(30, 41, 59, 0.9);
      --intro-text: #f8fafc;
      --intro-muted: #94a3b8;
      --intro-border: #334155;
      --badge-bg: #0f172a;
      --badge-border: #1e293b;
      --badge-text: #cbd5e1;
      --accent: #60a5fa;
    }
  }

  .intro-container {
    font-family: system-ui, -apple-system, sans-serif;
    background: var(--intro-bg);
    padding: 24px 32px;
    border-radius: 16px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.05), inset 0 1px 0 rgba(255,255,255,0.4);
    margin: 16px 0;
    border: 1px solid var(--intro-border);
  }

  .intro-header {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 16px;
  }

  .intro-title {
    margin: 0;
    font-size: 24px;
    font-weight: 800;
    color: var(--intro-text);
    letter-spacing: -0.5px;
  }

  /* Mini Pipeline */
  .mini-pipeline {
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    gap: 12px;
    background: var(--intro-card);
    padding: 12px 16px;
    border-radius: 8px;
    border: 1px solid var(--intro-border);
    font-size: 13px;
    font-weight: 600;
    color: var(--intro-text);
    margin-bottom: 24px;
  }

  .pipeline-step {
    display: flex;
    align-items: center;
    gap: 6px;
  }

  .pipeline-step i { color: var(--accent); }

  .pipeline-arrow {
    color: var(--intro-muted);
    font-size: 12px;
  }

  /* Content Sections */
  .intro-section { margin-bottom: 20px; }

  .section-title {
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 1px;
    color: var(--intro-muted);
    font-weight: 700;
    margin-bottom: 10px;
    display: block;
  }

  /* Badges para variables */
  .variables-grid {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
  }

  .var-badge {
    background: var(--badge-bg);
    border: 1px solid var(--badge-border);
    color: var(--badge-text);
    padding: 6px 12px;
    border-radius: 20px;
    font-size: 12px;
    font-weight: 500;
    transition: all 0.2s ease;
  }

  .var-badge:hover {
    border-color: var(--accent);
    color: var(--accent);
    transform: translateY(-1px);
  }

  /* Footer Specs */
  .specs-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
    gap: 16px;
    margin-top: 24px;
    padding-top: 20px;
    border-top: 1px dashed var(--intro-border);
  }

  .spec-item {
    font-size: 13px;
    color: var(--intro-text);
    line-height: 1.6;
  }

  .spec-item a {
    color: var(--accent);
    text-decoration: none;
    font-weight: 600;
  }

  .spec-item a:hover { text-decoration: underline; }

</style>

<div class="intro-container">
  <div class="intro-header">
    <span style="font-size: 28px;">🌤️</span>
    <h1 class="intro-title">Panel Meteorológico Interactivo — Argentina</h1>
  </div>

  <div class="mini-pipeline">
    <div class="pipeline-step"><i>☁️</i> Open-Meteo API</div>
    <div class="pipeline-arrow">➔</div>
    <div class="pipeline-step"><i>🐼</i> Pandas (Snapshot tabular)</div>
    <div class="pipeline-arrow">➔</div>
    <div class="pipeline-step"><i>⚡</i> Dashboard HTML/JS (Auto-refresh 60s)</div>
  </div>

  <div class="intro-section">
    <span class="section-title">Variables Monitoreadas</span>
    <div class="variables-grid">
      <div class="var-badge">🌡️ Temperatura</div>
      <div class="var-badge">🥶 Sensación Térmica</div>
      <div class="var-badge">💧 Humedad</div>
      <div class="var-badge">🌧️ Precipitación</div>
      <div class="var-badge">💨 Viento (Vel, Dir, Ráfagas)</div>
      <div class="var-badge">⏲️ Presión</div>
      <div class="var-badge">☀️ Índice UV</div>
      <div class="var-badge">📊 Código WMO</div>
      <div class="var-badge">🍃 Calidad del Aire (PM2.5, PM10, AQI)</div>
      <div class="var-badge">🌅 Amanecer / Atardecer</div>
    </div>
  </div>

  <div class="specs-grid">
    <div class="spec-item">
      <b>📡 Fuente de Datos:</b><br/>
      <a href="https://open-meteo.com" target="_blank">open-meteo.com</a> — API de código abierto, gratuita y sin necesidad de API Key.
    </div>
    <div class="spec-item">
      <b>🗺️ Motor de Mapas:</b><br/>
      Leaflet.js renderizando 5 capas base interactivas (CARTO, OSM, OpenTopoMap, Esri Satélite, CARTO Dark).
    </div>
    <div class="spec-item">
      <b>👨‍🎓 Alumnos:</b><br/>
      <a href="https://www.linkedin.com/in/gerardoaboulafia" target="_blank">Gerardo Aboulafia</a>,
      <a href="https://www.linkedin.com/in/santiago-arena-92a8191a2/" target="_blank">Santiago Arena</a>,
      <a href="https://www.linkedin.com/in/alvaro-hernandez-37021a37a" target="_blank">Álvaro Hernández</a>,
      <a href="https://www.linkedin.com/in/nicolas-masino/" target="_blank">Nicolás Masino</a>
    </div>
    <div class="spec-item">
      <b>👨‍🏫 Profesor:</b><br/>
      <a href="https://www.linkedin.com/in/julio-paredes-rojas-b0143752/" target="_blank">Julio Paredes Rojas</a>
    </div>
  </div>
</div>
"""

display(HTML(intro_html))

In [ ]:
# @title
"""Arquitectura de streaming (dinámica) con énfasis en Kafka.
Esta celda genera un diagrama HTML interactivo (Dark Mode + Tooltips + Simulación de Carga).
"""

from IPython.display import HTML, display

arch_html = """
<style>
  :root {
    --bg-main: linear-gradient(135deg, #e0e7ff 0%, #f1f5f9 100%);
    --card-bg: rgba(255, 255, 255, 0.85);
    --kafka-border: #f97316;
    --kafka-bg: rgba(255, 247, 237, 0.9);
    --text-main: #1e293b;
    --text-muted: #475569;
    --tooltip-bg: #1e293b;
    --tooltip-text: #f8fafc;
    --legend-bg: rgba(255,255,255,0.7);
  }

  @media (prefers-color-scheme: dark) {
    :root {
      --bg-main: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
      --card-bg: rgba(30, 41, 59, 0.85);
      --kafka-bg: rgba(67, 20, 7, 0.8);
      --text-main: #f1f5f9;
      --text-muted: #cbd5e1;
      --tooltip-bg: #f8fafc;
      --tooltip-text: #0f172a;
      --legend-bg: rgba(15, 23, 42, 0.7);
    }
  }

  .stream-container {
    font-family: system-ui, -apple-system, sans-serif;
    background: var(--bg-main);
    padding: 24px;
    border-radius: 16px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.15);
    margin: 16px 0;
    backdrop-filter: blur(10px);
    position: relative;
  }

  /* Header & Toggle Switch */
  .header-area {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 20px;
  }

  .stream-title {
    color: var(--text-main);
    font-size: 18px;
    font-weight: 700;
  }

  .toggle-container {
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 13px;
    color: var(--text-main);
    font-weight: 600;
  }

  .toggle-switch {
    position: relative;
    display: inline-block;
    width: 44px;
    height: 24px;
  }

  .toggle-switch input { opacity: 0; width: 0; height: 0; }

  .slider {
    position: absolute; cursor: pointer;
    top: 0; left: 0; right: 0; bottom: 0;
    background-color: #cbd5e1;
    transition: .4s; border-radius: 24px;
  }

  .slider:before {
    position: absolute; content: "";
    height: 18px; width: 18px; left: 3px; bottom: 3px;
    background-color: white;
    transition: .4s; border-radius: 50%;
  }

  input:checked + .slider { background-color: #ef4444; }
  input:checked + .slider:before { transform: translateX(20px); }

  /* Flow Area */
  .stream-flow {
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: center;
    gap: 12px;
  }

  .node-card {
    flex: 1 1 150px;
    background: var(--card-bg);
    border: 1px solid rgba(255,255,255,0.2);
    border-radius: 12px;
    padding: 16px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    min-height: 100px;
    position: relative;
    transition: all 0.3s ease;
    cursor: help;
  }

  .node-card:hover {
    transform: translateY(-4px);
    box-shadow: 0 12px 20px rgba(0,0,0,0.1);
  }

  .node-card h4 { margin: 0 0 8px 0; font-size: 14px; color: var(--text-main); }
  .node-card p { margin: 0; font-size: 12px; color: var(--text-muted); line-height: 1.5; }

  /* Tooltip logic */
  .tech-tooltip {
    position: absolute;
    bottom: 110%;
    left: 50%;
    transform: translateX(-50%);
    background: var(--tooltip-bg);
    color: var(--tooltip-text);
    padding: 8px 12px;
    border-radius: 6px;
    font-size: 11px;
    font-weight: 500;
    white-space: nowrap;
    opacity: 0;
    visibility: hidden;
    transition: all 0.2s ease;
    pointer-events: none;
    z-index: 10;
  }

  .tech-tooltip::after {
    content: ''; position: absolute; top: 100%; left: 50%;
    margin-left: -5px; border-width: 5px; border-style: solid;
    border-color: var(--tooltip-bg) transparent transparent transparent;
  }

  .node-card:hover .tech-tooltip {
    opacity: 1; visibility: visible; bottom: 120%;
  }

  /* Kafka Specifics */
  .kafka-node {
    border: 2px solid var(--kafka-border);
    background: var(--kafka-bg);
  }

  .flow-connector {
    display: flex; align-items: center; justify-content: center;
    width: 40px; height: 20px; position: relative;
  }

  .flow-dot {
    width: 6px; height: 6px;
    background-color: var(--kafka-border);
    border-radius: 50%; position: absolute;
    animation: flowData 2s infinite cubic-bezier(0.4, 0, 0.2, 1);
  }

  /* --- INTERACTIVE STATE (HIGH LOAD) --- */
  #stress-test:checked ~ .stream-flow .flow-connector.ingest .flow-dot {
    animation: flowData 0.3s infinite linear;
    background-color: #ef4444;
    box-shadow: 0 0 8px #ef4444;
  }

  #stress-test:checked ~ .stream-flow .kafka-node {
    border-color: #ef4444;
    box-shadow: 0 0 15px rgba(239, 68, 68, 0.3);
    animation: shake 0.5s infinite ease-in-out alternate;
  }

  #stress-test:checked ~ .stream-flow .flow-connector.ui .flow-dot {
    /* UI Flow remains steady, proving Kafka buffers the load! */
    animation: flowData 1.5s infinite cubic-bezier(0.4, 0, 0.2, 1);
    background-color: #22c55e;
  }

  /* Animations */
  @keyframes flowData {
    0% { left: 0; opacity: 0; transform: scale(0.5); }
    20% { opacity: 1; transform: scale(1.2); }
    80% { opacity: 1; transform: scale(1.2); }
    100% { left: 100%; opacity: 0; transform: scale(0.5); }
  }

  @keyframes shake {
    0% { transform: translateY(0); }
    100% { transform: translateY(-2px); }
  }

  .legend-box {
    margin-top: 24px; padding: 12px 16px;
    background: var(--legend-bg);
    border-left: 4px solid var(--kafka-border);
    border-radius: 4px 8px 8px 4px;
    font-size: 13px; color: var(--text-main);
  }

  @media (max-width: 900px) {
    .stream-flow { flex-direction: column; align-items: stretch; }
    .flow-connector { width: 100%; height: 30px; transform: rotate(90deg); }
  }
</style>

<div class="stream-container">

  <input type="checkbox" id="stress-test" style="display:none;">

  <div class="header-area">
    <div class="stream-title">⚡ Arquitectura Streaming End-to-End</div>
    <div class="toggle-container">
      <label for="stress-test">Simular Pico de Carga</label>
      <label class="toggle-switch">
        <input type="checkbox" id="stress-test-proxy" onclick="document.getElementById('stress-test').click()">
        <span class="slider"></span>
      </label>
    </div>
  </div>

  <div class="stream-flow">
    <div class="node-card">
      <div class="tech-tooltip">REST API • JSON • 15min Polling</div>
      <h4>🌐 1. Fuente</h4>
      <p>Open-Meteo API<br/>(Pronóstico y Calidad del aire)</p>
    </div>

    <div class="flow-connector ingest"><div class="flow-dot"></div></div>

    <div class="node-card kafka-node">
      <div class="tech-tooltip">confluent-kafka-python • Topic: weather.raw</div>
      <h4>📥 2. Kafka Ingest</h4>
      <p>Productor publica eventos en bruto</p>
    </div>

    <div class="flow-connector ingest"><div class="flow-dot"></div></div>

    <div class="node-card kafka-node">
      <div class="tech-tooltip">Partitions: 3 • Consumer Group: enrich-workers</div>
      <h4>⚙️ 3. Kafka Stream</h4>
      <p>Procesamiento y enriquecimiento de datos</p>
    </div>

    <div class="flow-connector ui"><div class="flow-dot"></div></div>

    <div class="node-card">
      <div class="tech-tooltip">WebSocket / Server-Sent Events (SSE)</div>
      <h4>🖥️ 4. Servicio UI</h4>
      <p>Consumer prepara el snapshot consolidado</p>
    </div>

    <div class="flow-connector ui"><div class="flow-dot"></div></div>

    <div class="node-card">
      <div class="tech-tooltip">Jupyter • Leaflet.js • Chart.js</div>
      <h4>📊 5. Dashboard</h4>
      <p>Visualización interactiva y auto-refresh</p>
    </div>
  </div>

  <div class="legend-box">
    <b>💡 Concepto clave:</b> Pasa el mouse sobre las tarjetas para ver los metadatos técnicos. Al activar el <b>Pico de Carga</b>, observarás cómo Kafka acelera su ingesta (rojo) pero el flujo hacia el dashboard (verde) se mantiene estable y controlado, demostrando su capacidad como buffer.
  </div>
</div>
"""

display(HTML(arch_html))

# Panel Meteorológico Interactivo — Argentina

```
Open-Meteo API  →  Pandas (snapshot tabular)  →  Dashboard HTML/JS (auto-refresh 60s)
```

**Variables monitoreadas:** temperatura, sensación térmica, humedad, precipitación, viento (velocidad, dirección, ráfagas), presión, UV, código de condición WMO, calidad del aire (PM2.5, PM10, AQI europeo), amanecer/atardecer.

**Fuente:** [open-meteo.com](https://open-meteo.com) — API gratuita, sin key.
**Mapa:** Leaflet + 5 capas base (CARTO, OSM, OpenTopoMap, Esri Satélite, CARTO Dark).


In [ ]:
# @title
# === CONFIGURACIÓN ===

CIUDADES = [
    {
        "nombre": "Buenos Aires",
        "lat": -34.6037,
        "lon": -58.3816
    },
    {
        "nombre": "Córdoba",
        "lat": -31.4201,
        "lon": -64.1888
    },
    {
        "nombre": "Rosario",
        "lat": -32.9468,
        "lon": -60.6393
    },
    {
        "nombre": "La Plata",
        "lat": -34.9215,
        "lon": -57.9545
    },
    {
        "nombre": "Mendoza",
        "lat": -32.8895,
        "lon": -68.8458
    },
    {
        "nombre": "Bariloche",
        "lat": -41.1335,
        "lon": -71.3103
    },
    {
        "nombre": "Mar del Plata",
        "lat": -38.0055,
        "lon": -57.5426
    },
    {
        "nombre": "Salta",
        "lat": -24.7821,
        "lon": -65.4232
    },
    {
        "nombre": "Ushuaia",
        "lat": -54.8019,
        "lon": -68.303
    }
]

CIUDAD_DEFAULT      = "Buenos Aires"
REFRESH_SEGUNDOS    = 60
ZONA_HORARIA        = "America/Argentina/Buenos_Aires"


In [ ]:
# @title 4. Snapshot enriquecido via Kafka: Open-Meteo -> Producer -> Topic -> Consumer -> Pandas
import json
import threading
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from IPython.display import display
from kafka import KafkaConsumer, KafkaProducer

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
KAFKA_POLL_SECONDS = 900

WEATHER_FIELDS = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "weather_code",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "surface_pressure",
    "uv_index",
    "is_day",
]
AIR_QUALITY_FIELDS = ["pm2_5", "pm10", "european_aqi"]

WMO = {
    0: "Despejado", 1: "Mayormente despejado", 2: "Parcialmente nublado", 3: "Nublado",
    45: "Niebla", 48: "Niebla helada",
    51: "Llovizna debil", 53: "Llovizna moderada", 55: "Llovizna fuerte",
    56: "Llovizna helada", 57: "Llovizna helada fuerte",
    61: "Lluvia debil", 63: "Lluvia moderada", 65: "Lluvia fuerte",
    66: "Lluvia helada", 67: "Lluvia helada fuerte",
    71: "Nieve debil", 73: "Nieve moderada", 75: "Nieve fuerte", 77: "Granos de nieve",
    80: "Chaparrones", 81: "Chaparrones moderados", 82: "Chaparrones fuertes",
    85: "Nevadas dispersas", 86: "Nevadas dispersas fuertes",
    95: "Tormenta", 96: "Tormenta con granizo", 99: "Tormenta con granizo fuerte",
}

DIRS_16 = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSO", "SO", "OSO", "O", "ONO", "NO", "NNO"]
producer_stop_event = threading.Event()
producer_thread = None
producer_lock = threading.Lock()


def compass(deg):
    if deg is None:
        return "--"
    return DIRS_16[int(((deg % 360) + 11.25) / 22.5) % 16]


def normalize_open_meteo_payload(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("latitude"), list):
        rows = []
        count = len(payload.get("latitude", []))
        current = payload.get("current", {}) or {}
        for index in range(count):
            rows.append({
                "latitude": payload["latitude"][index],
                "longitude": payload["longitude"][index],
                "current": {
                    key: values[index] if isinstance(values, list) and index < len(values) else None
                    for key, values in current.items()
                },
            })
        return rows
    if isinstance(payload, dict):
        return [payload]
    return []


def request_open_meteo(url, current_fields):
    params = {
        "latitude": ",".join(str(c["lat"]) for c in CIUDADES),
        "longitude": ",".join(str(c["lon"]) for c in CIUDADES),
        "current": ",".join(current_fields),
        "timezone": ZONA_HORARIA,
        "forecast_days": 1,
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return normalize_open_meteo_payload(response.json())


def fetch_enriched_city_snapshot():
    weather_rows = request_open_meteo(WEATHER_URL, WEATHER_FIELDS)
    try:
        air_rows = request_open_meteo(AIR_QUALITY_URL, AIR_QUALITY_FIELDS)
    except Exception as exc:
        print(f"[WARN] No se pudo obtener calidad del aire: {exc}")
        air_rows = [{} for _ in CIUDADES]

    snapshot_id = datetime.now(timezone.utc).isoformat()
    messages = []
    for index, ciudad in enumerate(CIUDADES):
        weather_current = (weather_rows[index].get("current", {}) if index < len(weather_rows) else {}) or {}
        air_current = (air_rows[index].get("current", {}) if index < len(air_rows) else {}) or {}
        wind_direction = weather_current.get("wind_direction_10m")
        weather_code = weather_current.get("weather_code")
        messages.append({
            "snapshot_id": snapshot_id,
            "source": "Open-Meteo enriched via Kafka",
            "ciudad": ciudad["nombre"],
            "lat": ciudad["lat"],
            "lon": ciudad["lon"],
            "current_time": weather_current.get("time"),
            "temperature_2m": weather_current.get("temperature_2m"),
            "apparent_temperature": weather_current.get("apparent_temperature"),
            "relative_humidity_2m": weather_current.get("relative_humidity_2m"),
            "precipitation": weather_current.get("precipitation"),
            "weather_code": weather_code,
            "weather_label": WMO.get(weather_code, "--"),
            "wind_speed_10m": weather_current.get("wind_speed_10m"),
            "wind_direction_10m": wind_direction,
            "wind_compass": compass(wind_direction),
            "wind_gusts_10m": weather_current.get("wind_gusts_10m"),
            "surface_pressure": weather_current.get("surface_pressure"),
            "uv_index": weather_current.get("uv_index"),
            "is_day": weather_current.get("is_day"),
            "pm2_5": air_current.get("pm2_5"),
            "pm10": air_current.get("pm10"),
            "european_aqi": air_current.get("european_aqi"),
        })
    return messages


def publish_snapshot_once():
    producer = KafkaProducer(
        bootstrap_servers=[BOOTSTRAP_SERVERS],
        key_serializer=lambda value: value.encode("utf-8"),
        value_serializer=lambda value: json.dumps(value, ensure_ascii=False).encode("utf-8"),
        acks="all",
        retries=3,
        linger_ms=20,
    )
    messages = fetch_enriched_city_snapshot()
    for message in messages:
        producer.send(TOPIC_NAME, key=message["ciudad"], value=message)
    producer.flush()
    producer.close()
    print(f"Producer publico {len(messages)} mensajes en '{TOPIC_NAME}'")
    return messages


def consume_latest_snapshot(timeout_ms=8000):
    consumer = KafkaConsumer(
        TOPIC_NAME,
        bootstrap_servers=[BOOTSTRAP_SERVERS],
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        consumer_timeout_ms=timeout_ms,
        key_deserializer=lambda raw: raw.decode("utf-8") if raw else None,
        value_deserializer=lambda raw: json.loads(raw.decode("utf-8")),
    )
    latest_by_city = {}
    for record in consumer:
        message = record.value
        message["_kafka_partition"] = record.partition
        message["_kafka_offset"] = record.offset
        latest_by_city[message["ciudad"]] = message
    consumer.close()

    missing = [city["nombre"] for city in CIUDADES if city["nombre"] not in latest_by_city]
    if missing:
        raise RuntimeError(f"Faltan ciudades en Kafka: {missing}")
    return [latest_by_city[city["nombre"]] for city in CIUDADES]


def start_producer_background(poll_seconds=KAFKA_POLL_SECONDS):
    global producer_thread
    if producer_thread is not None and producer_thread.is_alive():
        print("El producer continuo ya esta corriendo")
        return producer_thread

    def producer_loop():
        cycle = 0
        while not producer_stop_event.is_set():
            cycle += 1
            try:
                publish_snapshot_once()
                print(f"Ciclo Kafka {cycle} completado")
            except Exception as exc:
                print(f"[WARN] Ciclo Kafka {cycle} fallo: {exc}")
            producer_stop_event.wait(poll_seconds)
        print("Producer continuo detenido")

    producer_stop_event.clear()
    producer_thread = threading.Thread(target=producer_loop, daemon=True)
    producer_thread.start()
    print(f"Producer continuo iniciado cada {poll_seconds}s")
    return producer_thread


def stop_producer_background():
    producer_stop_event.set()
    print("Senal de stop enviada al producer continuo")


def build_snapshot_dataframe(messages):
    rows = []
    for message in messages:
        rows.append({
            "Ciudad": message["ciudad"],
            "Lat": message["lat"],
            "Lon": message["lon"],
            "Temp (C)": message.get("temperature_2m"),
            "Sens. (C)": message.get("apparent_temperature"),
            "HR (%)": message.get("relative_humidity_2m"),
            "Lluvia (mm)": message.get("precipitation"),
            "Viento (km/h)": message.get("wind_speed_10m"),
            "Dir.": message.get("wind_compass"),
            "Rafagas (km/h)": message.get("wind_gusts_10m"),
            "Presion (hPa)": message.get("surface_pressure"),
            "UV": message.get("uv_index"),
            "PM2.5": message.get("pm2_5"),
            "PM10": message.get("pm10"),
            "AQI EU": message.get("european_aqi"),
            "Condicion": message.get("weather_label"),
            "Kafka offset": message.get("_kafka_offset"),
        })
    return pd.DataFrame(rows)


published_messages = publish_snapshot_once()
kafka_snapshot = consume_latest_snapshot()
df = build_snapshot_dataframe(kafka_snapshot)

styled = (
    df.style
      .background_gradient(subset=["Temp (C)", "Sens. (C)"], cmap="RdYlBu_r")
      .background_gradient(subset=["Viento (km/h)", "Rafagas (km/h)"], cmap="Blues")
      .background_gradient(subset=["HR (%)"], cmap="Greens")
      .background_gradient(subset=["AQI EU"], cmap="Oranges")
      .format({
          "Lat": "{:.4f}", "Lon": "{:.4f}",
          "Temp (C)": "{:.1f}", "Sens. (C)": "{:.1f}",
          "HR (%)": "{:.0f}", "Lluvia (mm)": "{:.1f}",
          "Viento (km/h)": "{:.1f}", "Rafagas (km/h)": "{:.1f}",
          "Presion (hPa)": "{:.0f}", "UV": "{:.1f}",
          "PM2.5": "{:.1f}", "PM10": "{:.1f}", "AQI EU": "{:.0f}",
      }, na_rep="--")
      .set_properties(**{"text-align": "center"})
      .hide(axis="index")
)

display(styled)
print("Snapshot validado: Producer -> Kafka topic -> Consumer -> DataFrame enriquecido")


In [ ]:
# @title 5. Mantener Kafka produciendo en background mientras usas el tablero
# Ejecuta esta celda si quieres que el topic siga recibiendo snapshots.
# El dashboard HTML final de abajo se mantiene igual y sigue refrescando por su propio JavaScript.
producer_thread = start_producer_background(poll_seconds=KAFKA_POLL_SECONDS)


## Tablero interactivo en tiempo real

El siguiente tablero se actualiza solo cada **60 segundos** (configurable en la celda anterior).

- 🗺️ **Click** en cualquier ciudad o punto del mapa para cambiar la ubicación.
- 🛰️ Cambiá entre las **5 capas de mapa** (Carto, OSM, Topográfico, Satélite, Oscuro) en el control superior derecho.
- 🌗 **Tema claro/oscuro** con el botón en la cabecera (queda guardado entre sesiones).
- ⚠️ Si hay condiciones extremas (calor, frío, viento, lluvia, UV alto) aparece una **alerta** automática.
- 📈 **Sparkline** de la temperatura reciente (en memoria, hasta 30 puntos).


In [ ]:
# @title
# === DASHBOARD HTML/JS ===
# El template usa marcadores __CIUDADES_JSON__, __CIUDAD_DEFAULT__, __REFRESH__
# que se reemplazan con la config de la celda anterior.

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="es" data-theme="light">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Panel Meteorológico — Argentina</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
:root {
  --bg: #fafafa;
  --card: #ffffff;
  --card-alt: #f8fafc;
  --border: #e5e7eb;
  --text: #0f172a;
  --text-secondary: #475569;
  --text-muted: #94a3b8;
  --accent: #3b82f6;
  --accent-soft: rgba(59,130,246,0.08);
  --ok: #10b981;
  --warn: #f59e0b;
  --bad: #ef4444;
  --shadow: 0 1px 2px rgba(15,23,42,0.04), 0 1px 3px rgba(15,23,42,0.04);
}
[data-theme="dark"] {
  --bg: #0b1220;
  --card: #111a2c;
  --card-alt: #0f1729;
  --border: #1f2a40;
  --text: #f1f5f9;
  --text-secondary: #cbd5e1;
  --text-muted: #64748b;
  --accent: #60a5fa;
  --accent-soft: rgba(96,165,250,0.12);
  --shadow: 0 1px 2px rgba(0,0,0,0.3), 0 1px 3px rgba(0,0,0,0.2);
}
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; }
body {
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Inter, Roboto, sans-serif;
  background: var(--bg);
  color: var(--text);
  font-size: 13px;
  line-height: 1.5;
  -webkit-font-smoothing: antialiased;
  font-variant-numeric: tabular-nums;
  transition: background-color 0.2s ease, color 0.2s ease;
}
.container { max-width: 1400px; margin: 0 auto; padding: 24px; }

.header { display: flex; justify-content: space-between; align-items: flex-start; gap: 16px; margin-bottom: 20px; flex-wrap: wrap; }
.header h1 { font-size: 22px; font-weight: 600; margin: 0; letter-spacing: -0.01em; }
.subtitle { color: var(--text-secondary); font-size: 13px; margin-top: 4px; }
.header-actions { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; }

.status { display: flex; align-items: center; gap: 8px; color: var(--text-secondary); font-size: 12px; padding: 8px 12px; background: var(--card); border: 1px solid var(--border); border-radius: 8px; }
.status-dot { width: 7px; height: 7px; border-radius: 50%; background: var(--ok); flex-shrink: 0; }
.status-dot.error { background: var(--bad); }
.status-dot.loading { background: var(--warn); }

button.btn { background: var(--card); color: var(--text); border: 1px solid var(--border); border-radius: 8px; padding: 8px 14px; font-size: 13px; font-weight: 500; cursor: pointer; font-family: inherit; transition: border-color 0.15s, background-color 0.15s; }
button.btn:hover { border-color: var(--accent); }
button.btn.primary { background: var(--accent); color: #fff; border-color: var(--accent); }
button.btn.primary:hover { filter: brightness(1.05); }

.card { background: var(--card); border: 1px solid var(--border); border-radius: 12px; box-shadow: var(--shadow); padding: 20px; }
.card h2 { font-size: 11px; font-weight: 600; margin: 0 0 14px 0; color: var(--text-secondary); text-transform: uppercase; letter-spacing: 0.08em; }

.layout { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1.15fr); gap: 16px; margin-bottom: 16px; }

#map { height: 420px; border-radius: 8px; overflow: hidden; border: 1px solid var(--border); }
.leaflet-container { background: var(--bg); font: inherit; }

.metrics { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 10px; }
.metric { background: var(--card-alt); border: 1px solid var(--border); border-radius: 10px; padding: 14px; }
.metric-label { color: var(--text-secondary); font-size: 11px; text-transform: uppercase; letter-spacing: 0.06em; font-weight: 500; }
.metric-value { font-size: 26px; font-weight: 700; margin-top: 6px; color: var(--text); letter-spacing: -0.02em; line-height: 1.1; }
.metric-unit { font-size: 13px; color: var(--text-muted); font-weight: 500; margin-left: 2px; }
.metric-extra { color: var(--text-secondary); font-size: 11px; margin-top: 4px; }

.metric.span-2 { grid-column: span 2; }

.condition-row { display: flex; align-items: center; justify-content: space-between; padding: 14px 16px; background: var(--accent-soft); border-radius: 10px; margin-bottom: 12px; }
.condition-row .icon { font-size: 32px; }
.condition-row .label { font-size: 14px; font-weight: 600; }
.condition-row .sub { font-size: 12px; color: var(--text-secondary); }

.sparkline-wrap { display: flex; align-items: center; gap: 12px; margin-top: 6px; }
.sparkline-wrap svg { display: block; }

.compass { display: flex; align-items: center; gap: 14px; }
.compass svg { width: 60px; height: 60px; flex-shrink: 0; }

.daily-strip { display: grid; grid-template-columns: repeat(7, minmax(0, 1fr)); gap: 8px; }
.day-card { text-align: center; padding: 14px 6px; background: var(--card-alt); border: 1px solid var(--border); border-radius: 10px; }
.day-card .dow { font-size: 11px; color: var(--text-secondary); text-transform: uppercase; font-weight: 600; letter-spacing: 0.05em; }
.day-card .dom { font-size: 11px; color: var(--text-muted); margin-top: 2px; }
.day-card .icon { font-size: 26px; margin: 6px 0 4px; line-height: 1; }
.day-card .temps { font-size: 13px; }
.day-card .max { font-weight: 700; color: var(--text); }
.day-card .min { color: var(--text-muted); margin-left: 4px; }
.day-card .rain { color: var(--accent); font-size: 11px; margin-top: 4px; min-height: 14px; }

.table-wrap { overflow-x: auto; max-height: 480px; }
table { width: 100%; border-collapse: collapse; }
th { text-align: left; padding: 10px 12px; font-weight: 600; color: var(--text-secondary); font-size: 11px; text-transform: uppercase; letter-spacing: 0.06em; border-bottom: 1px solid var(--border); position: sticky; top: 0; background: var(--card); }
td { padding: 10px 12px; border-bottom: 1px solid var(--border); font-size: 13px; }
tr:last-child td { border-bottom: none; }
tr:hover td { background: var(--card-alt); }
td.now { background: var(--accent-soft); }

.badge { display: inline-block; padding: 3px 8px; border-radius: 6px; font-size: 11px; font-weight: 500; white-space: nowrap; }
.badge.ok { background: rgba(16,185,129,0.1); color: var(--ok); }
.badge.warn { background: rgba(245,158,11,0.12); color: var(--warn); }
.badge.bad { background: rgba(239,68,68,0.1); color: var(--bad); }
[data-theme="dark"] .badge.ok { background: rgba(16,185,129,0.15); }

.alert { background: rgba(239,68,68,0.06); border: 1px solid rgba(239,68,68,0.25); border-left: 3px solid var(--bad); border-radius: 10px; padding: 12px 16px; margin-bottom: 14px; color: var(--text); font-size: 13px; display: flex; align-items: center; gap: 10px; }
.alert .icon { font-size: 18px; }
.alert .alert-title { font-weight: 600; color: var(--bad); }

.three-col { display: grid; grid-template-columns: minmax(0, 2fr) minmax(0, 1fr) minmax(0, 1fr); gap: 16px; margin-bottom: 16px; }

.sun-bar { height: 6px; background: var(--card-alt); border-radius: 999px; overflow: hidden; margin: 12px 0 8px; position: relative; }
.sun-bar-fill { height: 100%; background: linear-gradient(90deg, #fb923c, #f59e0b, #fb923c); border-radius: 999px; }
.sun-times { display: flex; justify-content: space-between; gap: 12px; }
.sun-times .col .label { color: var(--text-muted); font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; }
.sun-times .col .value { font-size: 18px; font-weight: 600; margin-top: 2px; }

.aqi-num { font-size: 38px; font-weight: 700; line-height: 1; letter-spacing: -0.02em; }
.aqi-cat { font-size: 13px; margin-top: 4px; font-weight: 500; }
.aqi-pm { display: flex; gap: 18px; padding-top: 12px; border-top: 1px solid var(--border); margin-top: 14px; justify-content: center; }
.aqi-pm .pm-label { color: var(--text-muted); font-size: 11px; }
.aqi-pm .pm-value { font-weight: 600; font-size: 14px; }

.chart-wrap { position: relative; height: 240px; }

.footer { color: var(--text-muted); font-size: 11px; text-align: center; margin-top: 20px; padding-top: 16px; border-top: 1px solid var(--border); }
.footer a { color: var(--text-secondary); text-decoration: none; }
.footer a:hover { color: var(--accent); }

.leaflet-popup-content-wrapper { background: var(--card); color: var(--text); border-radius: 10px; box-shadow: var(--shadow); }
.leaflet-popup-tip { background: var(--card); }
.leaflet-popup-content { margin: 10px 14px; font-size: 13px; }
.leaflet-popup-content .pop-name { font-weight: 600; margin-bottom: 4px; }
.leaflet-popup-content .pop-temp { font-size: 18px; font-weight: 700; color: var(--accent); }
.leaflet-popup-content .pop-cond { color: var(--text-secondary); font-size: 12px; }
.leaflet-control-layers { background: var(--card) !important; color: var(--text) !important; border: 1px solid var(--border) !important; border-radius: 8px !important; box-shadow: var(--shadow) !important; }
.leaflet-control-layers-expanded { padding: 8px 12px !important; font-size: 13px !important; }
.leaflet-control-attribution { background: rgba(255,255,255,0.7) !important; font-size: 10px !important; }
[data-theme="dark"] .leaflet-control-attribution { background: rgba(17,26,44,0.85) !important; color: var(--text-muted) !important; }
[data-theme="dark"] .leaflet-control-attribution a { color: var(--text-secondary) !important; }
[data-theme="dark"] .leaflet-control-zoom a { background: var(--card) !important; color: var(--text) !important; border-color: var(--border) !important; }

@media (max-width: 1000px) {
  .layout, .three-col { grid-template-columns: 1fr; }
  .daily-strip { grid-template-columns: repeat(4, 1fr); }
}
@media (max-width: 600px) {
  .container { padding: 14px; }
  .metrics { grid-template-columns: repeat(2, 1fr); }
  .daily-strip { grid-template-columns: repeat(3, 1fr); }
  .header h1 { font-size: 19px; }
  #map { height: 320px; }
}
</style>
</head>
<body>
<div class="container">

  <div class="header">
    <div>
      <h1>Panel Meteorológico — Argentina</h1>
      <div class="subtitle" id="subtitle">Cargando…</div>
    </div>
    <div class="header-actions">
      <div class="status">
        <span class="status-dot" id="statusDot"></span>
        <span id="statusText">Inicializando</span>
      </div>
      <button class="btn primary" id="refreshBtn" type="button">Actualizar</button>
      <button class="btn" id="themeBtn" type="button" aria-label="Cambiar tema">🌙</button>
    </div>
  </div>

  <div id="alertContainer"></div>

  <div class="layout">
    <div class="card">
      <h2>Mapa interactivo</h2>
      <div id="map"></div>
      <div style="color: var(--text-muted); font-size: 11px; margin-top: 10px;">
        Click en cualquier ciudad o punto del mapa para cambiar la ubicación. Cambiá la capa con el control superior derecho.
      </div>
    </div>
    <div class="card">
      <h2>Condiciones actuales</h2>
      <div class="condition-row">
        <div style="display:flex;align-items:center;gap:14px;">
          <div class="icon" id="condIcon">—</div>
          <div>
            <div class="label" id="condLabel">—</div>
            <div class="sub" id="condSub">—</div>
          </div>
        </div>
        <div style="text-align:right;">
          <div style="font-size:34px;font-weight:700;letter-spacing:-0.02em;line-height:1;" id="condTemp">—</div>
          <div class="sub" id="condFeels">Sensación —</div>
        </div>
      </div>
      <div class="metrics" id="metrics"></div>
      <div class="sparkline-wrap" style="margin-top:14px;">
        <span style="color:var(--text-muted);font-size:11px;text-transform:uppercase;letter-spacing:0.05em;">Historial reciente</span>
        <svg id="sparkline" width="160" height="28" viewBox="0 0 160 28"></svg>
      </div>
    </div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <h2>Pronóstico próximas 24 horas</h2>
    <div class="chart-wrap"><canvas id="hourlyChart"></canvas></div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <h2>Pronóstico 7 días</h2>
    <div class="daily-strip" id="dailyStrip"></div>
  </div>

  <div class="three-col">
    <div class="card">
      <h2>Pronóstico horario detallado</h2>
      <div class="table-wrap">
        <table>
          <thead><tr>
            <th>Hora</th><th>Temp</th><th>Sens.</th><th>HR</th><th>Lluvia</th><th>Viento</th><th>Condición</th>
          </tr></thead>
          <tbody id="hourlyTable"></tbody>
        </table>
      </div>
    </div>
    <div class="card">
      <h2>Sol</h2>
      <div id="sunInfo"><div style="color:var(--text-muted);">—</div></div>
    </div>
    <div class="card">
      <h2>Calidad del aire</h2>
      <div id="airInfo"><div style="color:var(--text-muted);">—</div></div>
    </div>
  </div>

  <div class="footer">
    <div style="margin-bottom: 8px; font-size: 12px; color: var(--text-secondary);">
      <b>👨‍🎓 Alumnos:</b>
      <a href="https://www.linkedin.com/in/gerardoaboulafia" target="_blank" rel="noopener">Gerardo Aboulafia</a>,
      <a href="https://www.linkedin.com/in/santiago-arena-92a8191a2/" target="_blank" rel="noopener">Santiago Arena</a>,
      <a href="https://www.linkedin.com/in/alvaro-hernandez-37021a37a" target="_blank" rel="noopener">Álvaro Hernández</a>,
      <a href="https://www.linkedin.com/in/nicolas-masino/" target="_blank" rel="noopener">Nicolás Masino</a>
      <span style="margin: 0 10px; color: var(--border);">|</span>
      <b>👨‍🏫 Profesor:</b>
      <a href="https://www.linkedin.com/in/julio-paredes-rojas-b0143752/" target="_blank" rel="noopener">Julio Paredes Rojas</a>
    </div>
    Datos: <a href="https://open-meteo.com" target="_blank" rel="noopener">Open-Meteo</a> ·
    Mapa: <a href="https://leafletjs.com" target="_blank" rel="noopener">Leaflet</a> ·
    Tiles: OpenStreetMap, CARTO, OpenTopoMap, Esri
  </div>
</div>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<script>
// ===========================================================================
// CONFIG (inyectada desde Python)
// ===========================================================================
const CIUDADES = __CIUDADES_JSON__;
const CIUDAD_DEFAULT = "__CIUDAD_DEFAULT__";
const REFRESH_SEGUNDOS = __REFRESH__;
const TZ = "America/Argentina/Buenos_Aires";

// ===========================================================================
// ESTADO
// ===========================================================================
let activa = CIUDADES.find(c => c.nombre === CIUDAD_DEFAULT) || CIUDADES[0];
let historiaTemp = [];
let chart = null;
let countdown = REFRESH_SEGUNDOS;
let refreshTimer = null;
let countdownTimer = null;
let customMarker = null;
let cityMarkers = [];

// ===========================================================================
// HELPERS
// ===========================================================================
const fmt = (v, d = 1) => (v == null || isNaN(v)) ? "—" : (+v).toFixed(d);
const fmt0 = v => fmt(v, 0);

const DIRS_16 = ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSO","SO","OSO","O","ONO","NO","NNO"];
function compass(deg) {
  if (deg == null) return "—";
  return DIRS_16[Math.floor(((deg % 360) + 11.25) / 22.5) % 16];
}

function wmoInterpret(code, isDay) {
  const day = isDay !== 0;
  const m = {
    0:  { l: "Despejado",                  i: day ? "☀️" : "🌙",  s: "ok"   },
    1:  { l: "Mayormente despejado",       i: day ? "🌤️" : "🌙",  s: "ok"   },
    2:  { l: "Parcialmente nublado",       i: day ? "⛅" : "☁️",  s: "ok"   },
    3:  { l: "Nublado",                    i: "☁️",               s: "ok"   },
    45: { l: "Niebla",                     i: "🌫️",              s: "warn" },
    48: { l: "Niebla con escarcha",        i: "🌫️",              s: "warn" },
    51: { l: "Llovizna débil",             i: "🌦️",              s: "warn" },
    53: { l: "Llovizna moderada",          i: "🌦️",              s: "warn" },
    55: { l: "Llovizna intensa",           i: "🌧️",              s: "bad"  },
    56: { l: "Llovizna helada débil",      i: "🌧️",              s: "bad"  },
    57: { l: "Llovizna helada intensa",    i: "🌧️",              s: "bad"  },
    61: { l: "Lluvia débil",               i: "🌧️",              s: "warn" },
    63: { l: "Lluvia moderada",            i: "🌧️",              s: "bad"  },
    65: { l: "Lluvia intensa",             i: "🌧️",              s: "bad"  },
    66: { l: "Lluvia helada débil",        i: "🌧️",              s: "bad"  },
    67: { l: "Lluvia helada intensa",      i: "🌧️",              s: "bad"  },
    71: { l: "Nieve débil",                i: "🌨️",              s: "warn" },
    73: { l: "Nieve moderada",             i: "🌨️",              s: "bad"  },
    75: { l: "Nieve intensa",              i: "❄️",               s: "bad"  },
    77: { l: "Granos de nieve",            i: "❄️",               s: "warn" },
    80: { l: "Chaparrones débiles",        i: "🌦️",              s: "warn" },
    81: { l: "Chaparrones moderados",      i: "🌧️",              s: "bad"  },
    82: { l: "Chaparrones violentos",      i: "⛈️",              s: "bad"  },
    85: { l: "Nevadas dispersas débiles",  i: "🌨️",              s: "warn" },
    86: { l: "Nevadas dispersas intensas", i: "🌨️",              s: "bad"  },
    95: { l: "Tormenta",                   i: "⛈️",              s: "bad"  },
    96: { l: "Tormenta con granizo",       i: "⛈️",              s: "bad"  },
    99: { l: "Tormenta con granizo fuerte",i: "⛈️",              s: "bad"  },
  };
  return m[code] || { l: "Variable", i: "🌥️", s: "warn" };
}

function aqiCategory(aqi) {
  if (aqi == null) return { label: "Sin datos", color: "var(--text-muted)" };
  if (aqi <= 20)  return { label: "Buena",            color: "#10b981" };
  if (aqi <= 40)  return { label: "Aceptable",        color: "#84cc16" };
  if (aqi <= 60)  return { label: "Moderada",         color: "#f59e0b" };
  if (aqi <= 80)  return { label: "Mala",             color: "#ef4444" };
  if (aqi <= 100) return { label: "Muy mala",         color: "#a21caf" };
  return            { label: "Extremadamente mala", color: "#7f1d1d" };
}

function uvCategory(uv) {
  if (uv == null) return { label: "—", color: "var(--text-muted)" };
  if (uv < 3)  return { label: "Bajo",      color: "#10b981" };
  if (uv < 6)  return { label: "Moderado",  color: "#f59e0b" };
  if (uv < 8)  return { label: "Alto",      color: "#fb923c" };
  if (uv < 11) return { label: "Muy alto",  color: "#ef4444" };
  return         { label: "Extremo",   color: "#a21caf" };
}

// ===========================================================================
// MAPA — usa SOLO URLs de tiles verificadas (allowlist del plan)
// ===========================================================================
const baseLayers = {
  "Claro (CARTO)": L.tileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png", {
    attribution: "&copy; OpenStreetMap, &copy; CARTO",
    subdomains: "abcd", maxZoom: 19,
  }),
  "OpenStreetMap": L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png", {
    attribution: "&copy; OpenStreetMap contributors",
    subdomains: "abc", maxZoom: 19,
  }),
  "Topográfico": L.tileLayer("https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png", {
    attribution: "Map data: &copy; OpenStreetMap, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)",
    subdomains: "abc", maxZoom: 17,
  }),
  "Satélite (Esri)": L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}", {
    attribution: "Tiles &copy; Esri",
    maxZoom: 19,
  }),
  "Oscuro (CARTO)": L.tileLayer("https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png", {
    attribution: "&copy; OpenStreetMap, &copy; CARTO",
    subdomains: "abcd", maxZoom: 19,
  }),
};

const map = L.map("map", { zoomControl: true });
baseLayers["Claro (CARTO)"].addTo(map);
let currentBaseName = "Claro (CARTO)";
L.control.layers(baseLayers, null, { position: "topright", collapsed: true }).addTo(map);
map.on("baselayerchange", (e) => { currentBaseName = e.name; });

// Marcadores de ciudades pre-cargados
cityMarkers = CIUDADES.map(c => {
  const m = L.marker([c.lat, c.lon]).addTo(map);
  m._ciudad = c;
  m.bindPopup(`<div class="pop-name">${c.nombre}</div><div class="pop-cond">Cargando…</div>`);
  m.on("click", () => selectCity(c));
  return m;
});

// Ajustar vista a todas las ciudades
const group = L.featureGroup(cityMarkers);
map.fitBounds(group.getBounds().pad(0.12));

// Click en mapa vacío → marcador personalizado
map.on("click", function (e) {
  // Si fue click sobre marcador, no hacer nada (Leaflet ya lo maneja)
  if (e.originalEvent && e.originalEvent.target && e.originalEvent.target.closest(".leaflet-marker-icon")) return;
  const lat = +e.latlng.lat.toFixed(4);
  const lon = +e.latlng.lng.toFixed(4);
  if (customMarker) map.removeLayer(customMarker);
  customMarker = L.marker([lat, lon]).addTo(map)
    .bindPopup(`<div class="pop-name">📍 Punto personalizado</div><div class="pop-cond">${lat}, ${lon}</div>`)
    .openPopup();
  selectCity({ nombre: "Punto personalizado", lat, lon, custom: true });
});

// ===========================================================================
// TEMA
// ===========================================================================
const themeBtn = document.getElementById("themeBtn");
function setTheme(t) {
  document.documentElement.setAttribute("data-theme", t);
  try { localStorage.setItem("dashboard-theme", t); } catch (e) {}
  themeBtn.textContent = t === "light" ? "🌙" : "☀️";
  themeBtn.setAttribute("aria-label", t === "light" ? "Cambiar a tema oscuro" : "Cambiar a tema claro");
  // Switch map base sólo si el usuario está en una de las dos capas CARTO
  // (preserva su elección si eligió satélite, OSM, topo, etc.)
  if (typeof map !== "undefined" && (currentBaseName === "Claro (CARTO)" || currentBaseName === "Oscuro (CARTO)")) {
    const target = t === "dark" ? "Oscuro (CARTO)" : "Claro (CARTO)";
    if (target !== currentBaseName) {
      map.removeLayer(baseLayers[currentBaseName]);
      baseLayers[target].addTo(map);
      currentBaseName = target;
    }
  }
  if (chart) { renderHourly._lastFr && renderHourly(renderHourly._lastFr); }
}
let savedTheme = "light";
try { savedTheme = localStorage.getItem("dashboard-theme") || "light"; } catch (e) {}
setTheme(savedTheme);
themeBtn.addEventListener("click", () => {
  const cur = document.documentElement.getAttribute("data-theme");
  setTheme(cur === "light" ? "dark" : "light");
});

// ===========================================================================
// SELECCIÓN DE CIUDAD + CARGA
// ===========================================================================
function selectCity(c) {
  activa = c;
  historiaTemp = [];
  document.getElementById("subtitle").textContent =
    `${c.nombre} · ${(+c.lat).toFixed(4)}, ${(+c.lon).toFixed(4)}`;
  setStatus("loading", "Cargando…");
  cargarDatos();
  // Centrar mapa en selección sin alterar zoom mucho
  map.setView([c.lat, c.lon], Math.max(map.getZoom(), 5), { animate: true });
}

function setStatus(state, text) {
  const dot = document.getElementById("statusDot");
  dot.classList.remove("error", "loading");
  if (state === "error") dot.classList.add("error");
  if (state === "loading") dot.classList.add("loading");
  document.getElementById("statusText").textContent = text;
}

async function cargarDatos() {
  const lat = activa.lat, lon = activa.lon;
  const tzEnc = encodeURIComponent(TZ);
  const fcUrl = `https://api.open-meteo.com/v1/forecast`
    + `?latitude=${lat}&longitude=${lon}`
    + `&current=temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m,surface_pressure,cloud_cover,uv_index,is_day`
    + `&hourly=temperature_2m,apparent_temperature,relative_humidity_2m,precipitation_probability,precipitation,weather_code,wind_speed_10m,wind_direction_10m`
    + `&daily=weather_code,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,sunrise,sunset,uv_index_max`
    + `&forecast_days=7&timezone=${tzEnc}`;

  const aqUrl = `https://air-quality-api.open-meteo.com/v1/air-quality`
    + `?latitude=${lat}&longitude=${lon}`
    + `&current=pm10,pm2_5,european_aqi&timezone=${tzEnc}`;

  try {
    const [fr, ar] = await Promise.all([
      fetch(fcUrl).then(r => r.ok ? r.json() : Promise.reject("forecast " + r.status)),
      fetch(aqUrl).then(r => r.ok ? r.json() : { current: {} }).catch(() => ({ current: {} })),
    ]);

    renderActual(fr);
    renderHourly(fr);
    renderDaily(fr);
    renderSun(fr);
    renderAir(ar);
    renderAlerts(fr);
    actualizarPopupsCiudades(fr);

    // Sparkline
    const t = fr.current && fr.current.temperature_2m;
    if (t != null) {
      historiaTemp.push(t);
      if (historiaTemp.length > 30) historiaTemp.shift();
      renderSparkline();
    }

    countdown = REFRESH_SEGUNDOS;
    setStatus("ok", `Última actualización ${new Date().toLocaleTimeString("es-AR")} · próxima en ${countdown}s`);
  } catch (err) {
    console.error("Fetch error:", err);
    setStatus("error", "Error al cargar datos · reintentando…");
  }
}

// Actualiza el popup del marcador correspondiente con la temperatura más reciente
function actualizarPopupsCiudades(fr) {
  const cur = (fr && fr.current) || {};
  const wmo = wmoInterpret(cur.weather_code, cur.is_day);
  const t = cur.temperature_2m;
  const w = cur.wind_speed_10m;
  cityMarkers.forEach(m => {
    if (m._ciudad.nombre === activa.nombre) {
      m.setPopupContent(
        `<div class="pop-name">${m._ciudad.nombre}</div>`
        + `<div class="pop-temp">${fmt(t)} °C ${wmo.i}</div>`
        + `<div class="pop-cond">${wmo.l} · viento ${fmt0(w)} km/h</div>`
      );
    }
  });
}

// ===========================================================================
// RENDERS
// ===========================================================================
function renderActual(fr) {
  const c = (fr && fr.current) || {};
  const wmo = wmoInterpret(c.weather_code, c.is_day);
  document.getElementById("condIcon").textContent = wmo.i;
  document.getElementById("condLabel").textContent = wmo.l;
  document.getElementById("condSub").textContent =
    `Nubosidad ${fmt0(c.cloud_cover)}% · ${c.is_day ? "día" : "noche"}`;
  document.getElementById("condTemp").innerHTML =
    `${fmt(c.temperature_2m)}<span class="metric-unit">°C</span>`;
  document.getElementById("condFeels").textContent = `Sensación ${fmt(c.apparent_temperature)} °C`;

  const uvCat = uvCategory(c.uv_index);
  const compassDir = compass(c.wind_direction_10m);

  const cards = [
    { label: "Humedad",      value: fmt0(c.relative_humidity_2m), unit: "%",     extra: null },
    { label: "Lluvia",       value: fmt(c.precipitation),         unit: "mm",    extra: c.rain != null ? `Rain ${fmt(c.rain)} mm` : null },
    { label: "Viento",       value: fmt0(c.wind_speed_10m),       unit: "km/h",  extra: `Ráfagas ${fmt0(c.wind_gusts_10m)} km/h` },
    { label: "Dirección",    value: compassDir,                   unit: "",      extra: c.wind_direction_10m != null ? `${fmt0(c.wind_direction_10m)}°` : null, compass: c.wind_direction_10m },
    { label: "Presión",      value: fmt0(c.surface_pressure),     unit: "hPa",   extra: null },
    { label: "Índice UV",    value: fmt(c.uv_index),              unit: "",      extra: `<span style="color:${uvCat.color};font-weight:600;">${uvCat.label}</span>` },
  ];

  document.getElementById("metrics").innerHTML = cards.map(m => {
    if (m.compass != null && !isNaN(m.compass)) {
      const ang = m.compass;
      return `<div class="metric"><div class="metric-label">${m.label}</div>
        <div class="compass" style="margin-top:6px;">
          <svg viewBox="0 0 100 100" aria-hidden="true">
            <circle cx="50" cy="50" r="46" fill="none" stroke="var(--border)" stroke-width="1"/>
            <text x="50" y="14" text-anchor="middle" font-size="10" fill="var(--text-muted)">N</text>
            <text x="50" y="92" text-anchor="middle" font-size="10" fill="var(--text-muted)">S</text>
            <text x="12" y="54" text-anchor="middle" font-size="10" fill="var(--text-muted)">O</text>
            <text x="88" y="54" text-anchor="middle" font-size="10" fill="var(--text-muted)">E</text>
            <g transform="rotate(${ang} 50 50)">
              <polygon points="50,18 44,52 50,46 56,52" fill="var(--accent)"/>
              <polygon points="50,82 44,52 50,58 56,52" fill="var(--text-muted)" opacity="0.5"/>
            </g>
          </svg>
          <div>
            <div class="metric-value" style="font-size:22px;">${m.value}</div>
            <div class="metric-extra">${m.extra || ""}</div>
          </div>
        </div></div>`;
    }
    return `<div class="metric">
      <div class="metric-label">${m.label}</div>
      <div class="metric-value">${m.value}<span class="metric-unit"> ${m.unit}</span></div>
      ${m.extra ? `<div class="metric-extra">${m.extra}</div>` : ""}
    </div>`;
  }).join("");
}

function renderHourly(fr) {
  renderHourly._lastFr = fr;
  const h = (fr && fr.hourly) || {};
  const times = h.time || [];
  if (times.length === 0) return;

  // Encontrar el índice de la hora actual (o la siguiente más cercana)
  const now = new Date();
  let idx = times.findIndex(t => new Date(t) >= now);
  if (idx < 0) idx = 0;
  // Mostrar 24 horas desde ahora
  const end = Math.min(idx + 24, times.length);
  const sl = (arr) => (arr || []).slice(idx, end);
  const labels = sl(times).map(t => t.slice(11, 16));
  const temps  = sl(h.temperature_2m);
  const apps   = sl(h.apparent_temperature);
  const probs  = sl(h.precipitation_probability);
  const hums   = sl(h.relative_humidity_2m);
  const winds  = sl(h.wind_speed_10m);
  const dirs   = sl(h.wind_direction_10m);
  const codes  = sl(h.weather_code);

  // CHART
  const isDark = document.documentElement.getAttribute("data-theme") === "dark";
  const tickColor = isDark ? "#94a3b8" : "#64748b";
  const gridColor = isDark ? "rgba(148,163,184,0.12)" : "rgba(15,23,42,0.06)";

  if (chart) chart.destroy();
  const ctx = document.getElementById("hourlyChart").getContext("2d");
  chart = new Chart(ctx, {
    type: "line",
    data: {
      labels,
      datasets: [
        {
          label: "Temperatura (°C)",
          data: temps,
          borderColor: "#ef4444",
          backgroundColor: "rgba(239,68,68,0.08)",
          tension: 0.35,
          yAxisID: "y",
          pointRadius: 0,
          pointHoverRadius: 4,
          borderWidth: 2,
          fill: true,
        },
        {
          label: "Probabilidad de lluvia (%)",
          data: probs,
          borderColor: "#3b82f6",
          backgroundColor: "rgba(59,130,246,0.12)",
          tension: 0.35,
          yAxisID: "y1",
          pointRadius: 0,
          pointHoverRadius: 4,
          borderWidth: 2,
          fill: true,
        },
      ],
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      interaction: { intersect: false, mode: "index" },
      plugins: {
        legend: { position: "bottom", labels: { color: tickColor, boxWidth: 12, font: { size: 12 } } },
        tooltip: {
          backgroundColor: isDark ? "#0b1220" : "#ffffff",
          titleColor: isDark ? "#f1f5f9" : "#0f172a",
          bodyColor: isDark ? "#cbd5e1" : "#475569",
          borderColor: isDark ? "#1f2a40" : "#e5e7eb",
          borderWidth: 1, padding: 10,
        },
      },
      scales: {
        x: { ticks: { color: tickColor, maxRotation: 0, autoSkipPadding: 12 }, grid: { display: false } },
        y: { position: "left", ticks: { color: tickColor, callback: v => v + "°" }, grid: { color: gridColor } },
        y1: { position: "right", min: 0, max: 100, ticks: { color: tickColor, callback: v => v + "%" }, grid: { display: false } },
      },
    },
  });

  // TABLA
  const tbody = document.getElementById("hourlyTable");
  tbody.innerHTML = labels.map((t, i) => {
    const wmo = wmoInterpret(codes[i], 1);
    const isFirst = i === 0;
    return `<tr>
      <td class="${isFirst ? "now" : ""}"><b>${t}</b></td>
      <td><b>${fmt(temps[i])}°</b></td>
      <td>${fmt(apps[i])}°</td>
      <td>${fmt0(hums[i])}%</td>
      <td>${fmt0(probs[i])}%</td>
      <td>${fmt0(winds[i])} <span style="color:var(--text-muted);font-size:11px;">${compass(dirs[i])}</span></td>
      <td><span class="badge ${wmo.s}">${wmo.i} ${wmo.l}</span></td>
    </tr>`;
  }).join("");
}

function renderDaily(fr) {
  const d = (fr && fr.daily) || {};
  const t = d.time || [];
  if (t.length === 0) return;
  const html = t.map((dt, i) => {
    const date = new Date(dt + "T12:00:00");
    const dow = date.toLocaleDateString("es-AR", { weekday: "short" });
    const dom = date.toLocaleDateString("es-AR", { day: "numeric", month: "short" });
    const wmo = wmoInterpret((d.weather_code || [])[i], 1);
    const max = (d.temperature_2m_max || [])[i];
    const min = (d.temperature_2m_min || [])[i];
    const rain = (d.precipitation_sum || [])[i];
    const prob = (d.precipitation_probability_max || [])[i];
    return `<div class="day-card">
      <div class="dow">${dow}</div>
      <div class="dom">${dom}</div>
      <div class="icon" title="${wmo.l}">${wmo.i}</div>
      <div class="temps"><span class="max">${fmt0(max)}°</span><span class="min">${fmt0(min)}°</span></div>
      <div class="rain">${rain != null && rain > 0 ? `${rain.toFixed(1)} mm` : (prob != null ? `${fmt0(prob)}%` : "—")}</div>
    </div>`;
  }).join("");
  document.getElementById("dailyStrip").innerHTML = html;
}

function renderSun(fr) {
  const d = (fr && fr.daily) || {};
  const sr = (d.sunrise || [])[0];
  const ss = (d.sunset  || [])[0];
  if (!sr || !ss) {
    document.getElementById("sunInfo").innerHTML = '<div style="color:var(--text-muted);">Sin datos</div>';
    return;
  }
  // Tiempos vienen sin offset (timezone aplicado por API), pero new Date() local. Parseamos a HH:MM.
  const srHM = sr.slice(11, 16);
  const ssHM = ss.slice(11, 16);
  const srD = new Date(sr);
  const ssD = new Date(ss);
  const dayMin = (ssD - srD) / 60000;
  const hrs = Math.floor(dayMin / 60);
  const mins = Math.floor(dayMin % 60);
  const now = new Date();
  let pct = ((now - srD) / (ssD - srD)) * 100;
  pct = Math.max(0, Math.min(100, pct));
  const uvMax = (d.uv_index_max || [])[0];
  const uvCat = uvCategory(uvMax);

  document.getElementById("sunInfo").innerHTML = `
    <div class="sun-times">
      <div class="col"><div class="label">🌅 Amanecer</div><div class="value">${srHM}</div></div>
      <div class="col" style="text-align:right;"><div class="label">🌇 Atardecer</div><div class="value">${ssHM}</div></div>
    </div>
    <div class="sun-bar"><div class="sun-bar-fill" style="width:${pct.toFixed(1)}%"></div></div>
    <div style="display:flex;justify-content:space-between;color:var(--text-secondary);font-size:12px;">
      <span>Duración: ${hrs}h ${mins}m</span>
      <span>UV máx: <b style="color:${uvCat.color}">${fmt(uvMax)} (${uvCat.label})</b></span>
    </div>
  `;
}

function renderAir(ar) {
  const c = (ar && ar.current) || {};
  const aqi = c.european_aqi;
  const pm25 = c.pm2_5;
  const pm10 = c.pm10;
  const cat = aqiCategory(aqi);
  document.getElementById("airInfo").innerHTML = `
    <div style="text-align:center;">
      <div class="aqi-num" style="color:${cat.color};">${fmt0(aqi)}</div>
      <div class="aqi-cat" style="color:${cat.color};">${cat.label}</div>
      <div style="color:var(--text-muted);font-size:11px;margin-top:4px;">European AQI</div>
    </div>
    <div class="aqi-pm">
      <div style="text-align:center;">
        <div class="pm-label">PM 2.5</div>
        <div class="pm-value">${fmt(pm25)} <span style="color:var(--text-muted);font-size:11px;">μg/m³</span></div>
      </div>
      <div style="text-align:center;">
        <div class="pm-label">PM 10</div>
        <div class="pm-value">${fmt(pm10)} <span style="color:var(--text-muted);font-size:11px;">μg/m³</span></div>
      </div>
    </div>
  `;
}

function renderAlerts(fr) {
  const c = (fr && fr.current) || {};
  const alerts = [];
  if (c.temperature_2m != null && c.temperature_2m >= 35)
    alerts.push({ icon: "🔥", title: "Calor extremo",  text: `Temperatura ${c.temperature_2m}°C` });
  if (c.temperature_2m != null && c.temperature_2m <= 2)
    alerts.push({ icon: "🥶", title: "Frío extremo",   text: `Temperatura ${c.temperature_2m}°C` });
  if (c.wind_speed_10m != null && c.wind_speed_10m >= 50)
    alerts.push({ icon: "💨", title: "Viento fuerte",  text: `Sostenido ${c.wind_speed_10m} km/h${c.wind_gusts_10m ? `, ráfagas ${c.wind_gusts_10m} km/h` : ""}` });
  if (c.wind_gusts_10m != null && c.wind_gusts_10m >= 70 && (c.wind_speed_10m || 0) < 50)
    alerts.push({ icon: "💨", title: "Ráfagas fuertes", text: `Hasta ${c.wind_gusts_10m} km/h` });
  if (c.precipitation != null && c.precipitation >= 5)
    alerts.push({ icon: "🌧️", title: "Lluvia intensa", text: `${c.precipitation} mm en la última hora` });
  if (c.uv_index != null && c.uv_index >= 8)
    alerts.push({ icon: "☀️", title: "UV muy alto",    text: `Índice UV ${c.uv_index}` });

  document.getElementById("alertContainer").innerHTML = alerts.map(a =>
    `<div class="alert"><span class="icon">${a.icon}</span><div><span class="alert-title">${a.title}</span> · ${a.text}</div></div>`
  ).join("");
}

function renderSparkline() {
  const svg = document.getElementById("sparkline");
  if (!svg) return;
  const W = 160, H = 28, P = 2;
  if (historiaTemp.length < 2) {
    svg.innerHTML = `<text x="${W/2}" y="${H/2}" text-anchor="middle" dominant-baseline="central" fill="var(--text-muted)" font-size="10">Recolectando…</text>`;
    return;
  }
  const min = Math.min(...historiaTemp);
  const max = Math.max(...historiaTemp);
  const r = max - min || 1;
  const pts = historiaTemp.map((v, i) => {
    const x = (i / (historiaTemp.length - 1)) * (W - 2 * P) + P;
    const y = H - ((v - min) / r) * (H - 2 * P) - P;
    return `${x.toFixed(1)},${y.toFixed(1)}`;
  });
  const last = pts[pts.length - 1].split(",");
  svg.innerHTML = `
    <polyline points="${pts.join(" ")}" fill="none" stroke="var(--accent)" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round"/>
    <circle cx="${last[0]}" cy="${last[1]}" r="2.5" fill="var(--accent)"/>
  `;
}

// ===========================================================================
// TIMERS
// ===========================================================================
document.getElementById("refreshBtn").addEventListener("click", () => {
  setStatus("loading", "Actualizando…");
  cargarDatos();
});

function startTimers() {
  if (refreshTimer) clearInterval(refreshTimer);
  if (countdownTimer) clearInterval(countdownTimer);
  refreshTimer = setInterval(cargarDatos, REFRESH_SEGUNDOS * 1000);
  countdownTimer = setInterval(() => {
    countdown = Math.max(0, countdown - 1);
    const dot = document.getElementById("statusDot");
    if (!dot.classList.contains("error") && !dot.classList.contains("loading")) {
      const t = document.getElementById("statusText");
      const last = (t.textContent.match(/Última actualización (\S+)/) || [])[1];
      if (last) {
        t.textContent = `Última actualización ${last} · próxima en ${countdown}s`;
      }
    }
  }, 1000);
}

// ===========================================================================
// INIT
// ===========================================================================
selectCity(activa);
startTimers();
</script>
</body>
</html>
"""

html = (HTML_TEMPLATE
    .replace("__CIUDADES_JSON__",  json.dumps(CIUDADES, ensure_ascii=False))
    .replace("__CIUDAD_DEFAULT__", CIUDAD_DEFAULT)
    .replace("__REFRESH__",        str(REFRESH_SEGUNDOS)))

out_path = Path("dashboard_meteo_interactivo.html")
out_path.write_text(html, encoding="utf-8")
print(f"Dashboard: {out_path.resolve()}  ({len(html):,} bytes)")

display(HTML(html))

In [ ]:
# @title Limpieza opcional: detener producer, Kafka y ZooKeeper
DETENER_SERVICIOS = False  # Cambia a True cuando termines de usar el notebook.

if DETENER_SERVICIOS:
    stop_producer_background()
    time.sleep(2)
    stop_kafka_stack()
    print("Producer, Kafka y ZooKeeper detenidos")
else:
    print("Servicios activos. Cambia DETENER_SERVICIOS a True para limpiar al finalizar.")
